In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# -*- coding: utf-8 -*-
# Heart Disease Dataset - Cleaning & Preparation (No warnings)
# Kaggle Notebook ready

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# ---------------------------
# 1. LOAD DATA
# ---------------------------
df = pd.read_csv('/kaggle/input/datasets/benafialoua/heartdisease/heart_disease.csv')  # adjust path if needed

print("Original shape:", df.shape)
print(df.head())
print(df.info())
print(df['target_binary'].value_counts())

# ---------------------------
# 2. REMOVE SYNTHETIC / ABNORMAL ROWS
# ---------------------------
# Keep only rows where age, trestbps, chol, thalach are integers (no decimal part)
critical_cols = ['age', 'trestbps', 'chol', 'thalach']
for col in critical_cols:
    df = df[df[col].notna()]

df = df[np.floor(df['age']) == df['age']]
df = df[np.floor(df['trestbps']) == df['trestbps']]
df = df[np.floor(df['chol']) == df['chol']]
df = df[np.floor(df['thalach']) == df['thalach']]

for col in critical_cols:
    df[col] = df[col].astype(int)

print("After removing synthetic rows:", df.shape)

# ---------------------------
# 3. HANDLE MISSING VALUES (if any)
# ---------------------------
# Instead of inplace=True, use direct assignment
num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
for col in num_cols:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)

cat_cols = ['cp', 'restecg', 'slope', 'ca', 'thal', 'sex', 'fbs', 'exang']
for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode()[0] if not df[col].mode().empty else 0
        df[col] = df[col].fillna(mode_val)

print("Missing values after imputation:\n", df.isnull().sum())

# ---------------------------
# 4. REMOVE OUTLIERS
# ---------------------------
df = df[(df['chol'] >= 100) & (df['chol'] <= 600)]
df = df[(df['trestbps'] >= 80) & (df['trestbps'] <= 200)]
df = df[(df['oldpeak'] >= 0) & (df['oldpeak'] <= 6)]

print("After outlier removal:", df.shape)

# ---------------------------
# 5. SEPARATE FEATURES AND TARGET
# ---------------------------
X = df.drop(columns=['num', 'target_binary'])
y = df['target_binary']

# ---------------------------
# 6. PREPROCESSING PIPELINE
# ---------------------------
numeric_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
categorical_features = ['cp', 'restecg', 'slope', 'ca', 'thal', 'sex', 'fbs', 'exang']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ])

X_processed = preprocessor.fit_transform(X)

feature_names = (numeric_features + 
                 list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)))
print("Number of features after encoding:", X_processed.shape[1])

# ---------------------------
# 7. SPLIT TRAIN / TEST
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True)}")

# ---------------------------
# 8. SAVE CLEANED DATA (OPTIONAL)
# ---------------------------
df_clean = df.copy()
df_clean.to_csv('/kaggle/working/heart_disease_cleaned.csv', index=False)

np.save('/kaggle/working/X_train.npy', X_train)
np.save('/kaggle/working/X_test.npy', X_test)
np.save('/kaggle/working/y_train.npy', y_train)
np.save('/kaggle/working/y_test.npy', y_test)

print("Data cleaning and preparation complete. Ready for modeling.")

In [ ]:
# -*- coding: utf-8 -*-
# Heart Disease Dataset - Exploratory Data Analysis (EDA) - Sans avertissements
# Kaggle Notebook ready

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# ---------------------------
# 1. LOAD CLEANED DATA
# ---------------------------
try:
    df = df_clean
    print("Utilisation de df_clean existant en mémoire.")
except NameError:
    df = pd.read_csv('/kaggle/working/heart_disease_cleaned.csv')
    print("Chargement des données nettoyées depuis le fichier.")

print("Forme :", df.shape)
print("\nPremières lignes :")
print(df.head())

# ---------------------------
# 2. INFOS DE BASE & VALEURS MANQUANTES
# ---------------------------
print("\n=== Informations sur le dataset ===")
df.info()

print("\n=== Valeurs manquantes ===")
missing = df.isnull().sum()
print(missing[missing > 0] if any(missing > 0) else "Aucune valeur manquante.")

print("\n=== Statistiques descriptives ===")
display(df.describe(include='all').round(2))

# ---------------------------
# 3. ANALYSE DE LA VARIABLE CIBLE
# ---------------------------
print("\n=== Distribution de la cible ===")
print(df['target_binary'].value_counts())
print(df['target_binary'].value_counts(normalize=True))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['target_binary'].value_counts().plot(kind='bar', ax=axes[0], color=['skyblue', 'salmon'])
axes[0].set_title('Effectifs de la maladie cardiaque')
axes[0].set_xticklabels(['Pas de maladie (0)', 'Maladie (1)'], rotation=0)
axes[0].set_ylabel('Fréquence')

df['target_binary'].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['skyblue', 'salmon'])
axes[1].set_title('Proportion')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

# ---------------------------
# 4. DISTRIBUTIONS DES VARIABLES NUMÉRIQUES
# ---------------------------
numerical_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()
for i, col in enumerate(numerical_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='steelblue')
    axes[i].set_title(f'Distribution de {col}')
axes[5].axis('off')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()
for i, col in enumerate(numerical_cols):
    sns.boxplot(y=df[col], ax=axes[i], color='lightgreen')
    axes[i].set_title(f'Boîte à moustaches de {col}')
axes[5].axis('off')
plt.tight_layout()
plt.show()

# ---------------------------
# 5. ANALYSE DES VARIABLES CATÉGORIELLES
# ---------------------------
categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']

fig, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.flatten()
for i, col in enumerate(categorical_cols):
    sns.countplot(data=df, x=col, hue='target_binary', ax=axes[i], palette='Set2')
    axes[i].set_title(f'{col} vs Maladie cardiaque')
    axes[i].legend(title='Maladie', labels=['Non', 'Oui'])
plt.tight_layout()
plt.show()

# ---------------------------
# 6. MATRICE DE CORRÉLATION
# ---------------------------
corr = df[numerical_cols + ['target_binary']].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matrice de corrélation (variables numériques + cible)')
plt.show()

print("\n=== Corrélation avec target_binary ===")
print(corr['target_binary'].sort_values(ascending=False))

# ---------------------------
# 7. DÉTECTION DES OUTLIERS (IQR)
# ---------------------------
print("\n=== Détection des outliers (méthode IQR) ===")
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.2f}%)")

# ---------------------------
# 8. RELATION NUMÉRIQUES VS CIBLE (version corrigée sans avertissements)
# ---------------------------
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()
for i, col in enumerate(numerical_cols):
    sns.boxplot(data=df, x='target_binary', y=col, ax=axes[i], hue='target_binary', legend=False, palette='Set3')
    axes[i].set_title(f'{col} selon le statut de la maladie')
    axes[i].set_xticks([0, 1])
    axes[i].set_xticklabels(['Pas de maladie', 'Maladie'])
axes[5].axis('off')
plt.tight_layout()
plt.show()

# ---------------------------
# 9. RÉSUMÉ DES PRINCIPALES OBSERVATIONS
# ---------------------------
print("\n=== Principales observations de l'EDA ===")
print("- Le jeu de données est équilibré (pas de déséquilibre sévère des classes).")
print("- 'thalach' (fréquence cardiaque max) est plus faible chez les patients malades.")
print("- 'oldpeak' (dépression du segment ST) est plus élevé dans le groupe malade.")
print("- 'cp' (type de douleur thoracique) : asymptomatique (4) est fortement associé à la maladie.")
print("- 'thal' : défaut réversible (7) et défaut fixé (6) indiquent un risque plus élevé.")
print("- Aucune valeur manquante après nettoyage.")
print("- Des outliers existent pour 'chol' et 'trestbps' mais restent dans des plages médicales réalistes.")

In [ ]:
# -*- coding: utf-8 -*-
# Heart Disease Dataset - Data Preparation
# Kaggle Notebook ready

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# ---------------------------
# 1. LOAD CLEANED DATA
# ---------------------------
try:
    df = df_clean  # from previous step
    print("Using existing df_clean from memory.")
except NameError:
    df = pd.read_csv('/kaggle/working/heart_disease_cleaned.csv')
    print("Loaded cleaned data from file.")

print("Initial shape:", df.shape)

# ---------------------------
# 2. REMOVE DUPLICATES (if any)
# ---------------------------
initial_len = len(df)
df = df.drop_duplicates()
print(f"Removed {initial_len - len(df)} duplicate rows. New shape: {df.shape}")

# ---------------------------
# 3. HANDLE REMAINING MISSING VALUES (safety)
# ---------------------------
# In our cleaned data there are none, but we ensure robustness
num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
for col in num_cols:
    if df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)

cat_cols = ['cp', 'restecg', 'slope', 'ca', 'thal', 'sex', 'fbs', 'exang']
for col in cat_cols:
    if df[col].isnull().any():
        df[col].fillna(df[col].mode()[0], inplace=True)

print("Missing values after final check:\n", df.isnull().sum().sum())

# ---------------------------
# 4. FEATURE ENGINEERING
# ---------------------------
# Create new features that might help classification

# 4.1 Age groups (binning)
df['age_group'] = pd.cut(df['age'], bins=[0, 40, 50, 60, 100], labels=['<40', '40-49', '50-59', '60+'])

# 4.2 Cholesterol to HDL ratio? Not available, but we can create a risk score: oldpeak * thalach
# High oldpeak and low thalach are bad, so product might be interesting
df['risk_score'] = df['oldpeak'] * (220 - df['thalach'])  # approximate

# 4.3 Blood pressure categories
df['bp_category'] = pd.cut(df['trestbps'], bins=[0, 120, 140, 200], labels=['Normal', 'Elevated', 'High'])

# 4.4 Interaction: age * cholesterol (as a risk proxy)
df['age_chol'] = df['age'] * df['chol'] / 1000  # scaled

# 4.5 Flag for max heart rate below 100 (bradycardia)
df['low_thalach'] = (df['thalach'] < 100).astype(int)

print("New features added:", ['age_group', 'risk_score', 'bp_category', 'age_chol', 'low_thalach'])

# ---------------------------
# 5. SEPARATE FEATURES AND TARGET
# ---------------------------
# We keep all original features plus engineered ones, but we must encode categoricals later.
# 'num' and 'target_binary' are excluded from X.
X = df.drop(columns=['num', 'target_binary'])
y = df['target_binary']

# Identify numeric and categorical columns (including new ones)
numeric_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'risk_score', 'age_chol', 'low_thalach']
categorical_features = ['cp', 'restecg', 'slope', 'ca', 'thal', 'sex', 'fbs', 'exang', 'age_group', 'bp_category']

# ---------------------------
# 6. PREPROCESSING PIPELINE (Normalization + One-Hot Encoding)
# ---------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ])

# Apply preprocessing to X
X_processed = preprocessor.fit_transform(X)

# Get feature names after encoding
feature_names = (numeric_features + 
                 list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)))
print(f"Number of features after encoding: {X_processed.shape[1]}")

# ---------------------------
# 7. TRAIN / TEST SPLIT (stratified)
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True)}")
print(f"Test class distribution:\n{y_test.value_counts(normalize=True)}")

# ---------------------------
# 8. SAVE PREPARED DATA FOR LATER USE
# ---------------------------
# Save the preprocessed arrays
np.save('/kaggle/working/X_train.npy', X_train)
np.save('/kaggle/working/X_test.npy', X_test)
np.save('/kaggle/working/y_train.npy', y_train)
np.save('/kaggle/working/y_test.npy', y_test)

# Also save the preprocessor object (to reuse for predictions on new data)
import joblib
joblib.dump(preprocessor, '/kaggle/working/preprocessor.pkl')

print("\nData preparation complete. Saved arrays and preprocessor.")
print("Ready for modeling.")

In [ ]:
# -*- coding: utf-8 -*-
# Heart Disease Dataset - Model Selection
# Kaggle Notebook ready

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
import joblib

# ---------------------------
# 1. LOAD PREPARED DATA
# ---------------------------
X_train = np.load('/kaggle/working/X_train.npy')
X_test = np.load('/kaggle/working/X_test.npy')
y_train = np.load('/kaggle/working/y_train.npy')
y_test = np.load('/kaggle/working/y_test.npy')

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train class distribution:", np.bincount(y_train))
print("Test class distribution:", np.bincount(y_test))

# ---------------------------
# 2. DEFINE MODELS TO COMPARE
# ---------------------------
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42)
}

# ---------------------------
# 3. TRAIN AND EVALUATE EACH MODEL
# ---------------------------
results = []
best_recall = 0
best_model_name = None
best_model = None

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_proba) if y_proba is not None else None
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-score': f1,
        'ROC-AUC': roc
    })
    
    print(f"  Accuracy: {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall: {rec:.4f}")
    print(f"  F1-score: {f1:.4f}")
    if roc:
        print(f"  ROC-AUC: {roc:.4f}")
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix - {name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
    
    # Keep best model based on Recall (to minimize false negatives)
    if rec > best_recall:
        best_recall = rec
        best_model_name = name
        best_model = model

# ---------------------------
# 4. COMPARISON TABLE
# ---------------------------
results_df = pd.DataFrame(results)
print("\n=== MODEL COMPARISON ===")
print(results_df.to_string(index=False))

# Highlight best recall
best_row = results_df[results_df['Recall'] == results_df['Recall'].max()]
print(f"\nBest model by Recall: {best_row['Model'].values[0]} (Recall = {best_row['Recall'].values[0]:.4f})")

# ---------------------------
# 5. SAVE THE BEST MODEL
# ---------------------------
joblib.dump(best_model, '/kaggle/working/best_model.pkl')
print(f"\nBest model ({best_model_name}) saved to /kaggle/working/best_model.pkl")

# ---------------------------
# 6. (OPTIONAL) FEATURE IMPORTANCE for tree-based models
# ---------------------------
if hasattr(best_model, 'feature_importances_'):
    # Load feature names (they were saved in previous step? we need to recompute or load)
    # For simplicity, we assume the preprocessor was saved and we can recover names.
    # But here we just show if available.
    try:
        # If you saved feature_names earlier, load them; otherwise skip.
        importances = best_model.feature_importances_
        # We don't have the names here, but you could load preprocessor to get them.
        print("\nFeature importances available but not displayed (no feature names loaded).")
    except:
        pass

print("\nModel selection complete. Ready for fine-tuning or deployment.")

In [ ]:
# -*- coding: utf-8 -*-
# Heart Disease Dataset - Étape 6 : Optimisation complète (avec GPU)
# Kaggle Notebook ready

import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, make_scorer
import xgboost as xgb
import joblib
import subprocess

# ---------------------------
# 1. CHARGEMENT ET NETTOYAGE
# ---------------------------
print("Chargement du fichier original...")
df = pd.read_csv('/kaggle/input/datasets/benafialoua/heartdisease/heart_disease.csv')

print("Shape original:", df.shape)

# Suppression des lignes synthétiques (valeurs non entières pour âge, etc.)
critical_cols = ['age', 'trestbps', 'chol', 'thalach']
for col in critical_cols:
    df = df[df[col].notna()]
df = df[np.floor(df['age']) == df['age']]
df = df[np.floor(df['trestbps']) == df['trestbps']]
df = df[np.floor(df['chol']) == df['chol']]
df = df[np.floor(df['thalach']) == df['thalach']]
for col in critical_cols:
    df[col] = df[col].astype(int)

print("Shape après suppression synthétique:", df.shape)

# Suppression des outliers (plages réalistes)
df = df[(df['chol'] >= 100) & (df['chol'] <= 600)]
df = df[(df['trestbps'] >= 80) & (df['trestbps'] <= 200)]
df = df[(df['oldpeak'] >= 0) & (df['oldpeak'] <= 6)]

print("Shape après outliers:", df.shape)

# ---------------------------
# 2. SÉPARATION FEATURES / TARGET
# ---------------------------
X_raw = df.drop(columns=['num', 'target_binary'])
y = df['target_binary']

# ---------------------------
# 3. PRÉPROCESSING (standardisation + encodage)
# ---------------------------
numeric_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
categorical_features = ['cp', 'restecg', 'slope', 'ca', 'thal', 'sex', 'fbs', 'exang']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ])

X_processed = preprocessor.fit_transform(X_raw)
print("Shape après preprocessing:", X_processed.shape)

# ---------------------------
# 4. SPLIT TRAIN/TEST
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Distribution classes train:", np.bincount(y_train))
print("Distribution classes test:", np.bincount(y_test))

# Métrique : rappel (recall) – priorité à la détection des malades
scorer = make_scorer(recall_score)

# ---------------------------
# 5. OPTIMISATION RÉGRESSION LOGISTIQUE
# ---------------------------
print("\n=== Optimisation de la Régression Logistique ===")
param_grid_lr = {
    'C': [0.01, 0.05, 0.1, 0.5, 1, 5, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga'],
    'class_weight': [None, 'balanced']
}

grid_lr = GridSearchCV(LogisticRegression(max_iter=2000, random_state=42),
                       param_grid_lr, cv=5, scoring=scorer, n_jobs=-1, verbose=1)
start = time.time()
grid_lr.fit(X_train, y_train)
print(f"\nGrid search LR terminée en {time.time()-start:.2f} sec")
print("Meilleurs paramètres LR:", grid_lr.best_params_)
print("Meilleur recall (CV):", grid_lr.best_score_)

best_lr = grid_lr.best_estimator_
y_pred_lr = best_lr.predict(X_test)
recall_lr = recall_score(y_test, y_pred_lr)
print(f"Recall sur test (LR optimisée): {recall_lr:.4f}")

# ---------------------------
# 6. OPTIMISATION XGBOOST AVEC GPU (si disponible)
# ---------------------------
print("\n=== Optimisation XGBoost ===")

# Détection GPU
gpu_available = False
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    gpu_available = (result.returncode == 0)
    print("GPU détecté : utilisation de 'device=cuda'" if gpu_available else "GPU non détecté, utilisation CPU")
except:
    print("Impossible de détecter GPU, utilisation CPU")

# Configuration de base XGBoost
xgb_base = {
    'tree_method': 'hist',
    'random_state': 42,
    'eval_metric': 'logloss',
    'n_jobs': -1
}
if gpu_available:
    xgb_base['device'] = 'cuda'

# Pour éviter l'avertissement de mismatch device, nous pouvons forcer l'utilisation de DMatrix
# mais RandomizedSearchCV gère automatiquement. On laisse faire.

param_dist_xgb = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [0, 0.1, 1]
}

xgb_model = xgb.XGBClassifier(**xgb_base)
random_search = RandomizedSearchCV(xgb_model, param_dist_xgb, n_iter=30, cv=3,
                                   scoring=scorer, n_jobs=-1, random_state=42, verbose=1)
start = time.time()
random_search.fit(X_train, y_train)
print(f"\nRecherche XGBoost terminée en {time.time()-start:.2f} sec")
print("Meilleurs paramètres XGBoost:", random_search.best_params_)
print("Meilleur recall (CV):", random_search.best_score_)

best_xgb = random_search.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)
recall_xgb = recall_score(y_test, y_pred_xgb)
print(f"Recall sur test (XGBoost): {recall_xgb:.4f}")

# ---------------------------
# 7. SÉLECTION DU MODÈLE FINAL
# ---------------------------
print("\n=== Comparaison finale ===")
print(f"Régression logistique optimisée : recall = {recall_lr:.4f}")
print(f"XGBoost optimisé                : recall = {recall_xgb:.4f}")

if recall_xgb > recall_lr:
    final_model = best_xgb
    model_name = "XGBoost"
else:
    final_model = best_lr
    model_name = "Régression logistique"

print(f"\nModèle final sélectionné : {model_name}")

# Sauvegarde des modèles
joblib.dump(final_model, '/kaggle/working/final_model_optimized.pkl')
joblib.dump(best_lr, '/kaggle/working/best_logistic_regression.pkl')
joblib.dump(best_xgb, '/kaggle/working/best_xgboost.pkl')
print("Modèles sauvegardés dans /kaggle/working/")

# ---------------------------
# 8. INTERPRÉTATION RAPIDE (optionnelle)
# ---------------------------
if model_name == "XGBoost":
    importances = final_model.feature_importances_
    top_idx = np.argsort(importances)[-10:][::-1]
    print("\nTop 10 indices des features importantes:", top_idx)
else:
    coefs = final_model.coef_[0]
    top_pos = np.argsort(coefs)[-5:][::-1]
    top_neg = np.argsort(coefs)[:5]
    print("\nTop 5 coefficients positifs (facteurs de risque) :", top_pos)
    print("Top 5 coefficients négatifs (facteurs protecteurs) :", top_neg)

print("\nOptimisation terminée. Passez à l'étape 7 pour l'évaluation finale détaillée.")

In [ ]:
# -*- coding: utf-8 -*-
# Heart Disease Dataset - Étape 7 : Évaluation du modèle final (corrigée)
# Kaggle Notebook ready

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)
from sklearn.model_selection import cross_val_score, StratifiedKFold
import joblib

# ---------------------------
# 1. CHARGEMENT DU MODÈLE FINAL ET DES DONNÉES
# ---------------------------
final_model = joblib.load('/kaggle/working/final_model_optimized.pkl')
print("Modèle chargé :", type(final_model).__name__)

try:
    X_test, y_test
    print("Données de test trouvées en mémoire.")
except NameError:
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    from sklearn.compose import ColumnTransformer
    
    df = pd.read_csv('/kaggle/input/heart-disease-dataset/heart_disease.csv')
    critical_cols = ['age', 'trestbps', 'chol', 'thalach']
    for col in critical_cols:
        df = df[df[col].notna()]
    df = df[np.floor(df['age']) == df['age']]
    df = df[np.floor(df['trestbps']) == df['trestbps']]
    df = df[np.floor(df['chol']) == df['chol']]
    df = df[np.floor(df['thalach']) == df['thalach']]
    for col in critical_cols:
        df[col] = df[col].astype(int)
    df = df[(df['chol'] >= 100) & (df['chol'] <= 600)]
    df = df[(df['trestbps'] >= 80) & (df['trestbps'] <= 200)]
    df = df[(df['oldpeak'] >= 0) & (df['oldpeak'] <= 6)]
    
    X_raw = df.drop(columns=['num', 'target_binary'])
    y = df['target_binary']
    
    numeric_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
    categorical_features = ['cp', 'restecg', 'slope', 'ca', 'thal', 'sex', 'fbs', 'exang']
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_features),
            ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
        ])
    X_processed = preprocessor.fit_transform(X_raw)
    X_train, X_test, y_train, y_test = train_test_split(
        X_processed, y, test_size=0.2, random_state=42, stratify=y
    )
    print("Données reconstruites.")

print(f"Taille du jeu de test : {X_test.shape[0]} échantillons")

# ---------------------------
# 2. PRÉDICTIONS ET MÉTRIQUES
# ---------------------------
y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1] if hasattr(final_model, "predict_proba") else None

print("\n=== Performances sur le test ===")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred):.4f}")
print(f"F1-score  : {f1_score(y_test, y_pred):.4f}")
if y_proba is not None:
    print(f"ROC-AUC   : {roc_auc_score(y_test, y_proba):.4f}")

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['Pas maladie (0)', 'Maladie (1)']))

# ---------------------------
# 3. MATRICE DE CONFUSION
# ---------------------------
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Prédit Sain', 'Prédit Malade'],
            yticklabels=['Réel Sain', 'Réel Malade'])
plt.title('Matrice de confusion - Modèle final')
plt.ylabel('Vérité terrain')
plt.xlabel('Prédiction')
plt.show()

# ---------------------------
# 4. COURBE ROC
# ---------------------------
if y_proba is not None:
    fpr, tpr, thresholds = roc_curve(y_test, y_proba)
    roc_auc = roc_auc_score(y_test, y_proba)
    plt.figure(figsize=(6,5))
    plt.plot(fpr, tpr, label=f'ROC (AUC = {roc_auc:.3f})', linewidth=2)
    plt.plot([0,1], [0,1], 'k--', linewidth=1)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Taux de faux positifs (1 - Spécificité)')
    plt.ylabel('Taux de vrais positifs (Recall)')
    plt.title('Courbe ROC - Modèle final')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.show()

# ---------------------------
# 5. VALIDATION CROISÉE (sur l'ensemble complet)
# ---------------------------
X_full = X_processed
y_full = y
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(final_model, X_full, y_full, cv=cv, scoring='recall')
print(f"\n=== Validation croisée 5-fold (rappel) ===")
print(f"Scores par fold : {cv_scores}")
print(f"Recall moyen    : {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

# ---------------------------
# 6. ANALYSE DES ERREURS
# ---------------------------
errors = (y_pred != y_test)
error_indices = np.where(errors)[0]
print(f"\n=== Analyse des erreurs ===")
print(f"Nombre total d'erreurs : {len(error_indices)} / {len(y_test)} ({len(error_indices)/len(y_test)*100:.1f}%)")

false_negatives = (y_pred == 0) & (y_test == 1)
false_positives = (y_pred == 1) & (y_test == 0)
print(f"Faux négatifs (malades prédits sains) : {np.sum(false_negatives)}")
print(f"Faux positifs (sains prédits malades) : {np.sum(false_positives)}")

# ---------------------------
# 7. INTERPRÉTATION SPÉCIFIQUE (sans avertissements)
# ---------------------------
if hasattr(final_model, 'coef_'):
    coefs = final_model.coef_[0]
    print("\n=== Interprétation des coefficients (régression logistique) ===")
    feature_names = numeric_features + list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features))
    coef_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': coefs})
    coef_df = coef_df.sort_values('Coefficient', ascending=False)
    print("Top 5 facteurs de risque (coefficients positifs) :")
    print(coef_df.head(5).to_string(index=False))
    print("\nTop 5 facteurs protecteurs (coefficients négatifs) :")
    print(coef_df.tail(5).to_string(index=False))
    
    # Graphique corrigé : utilisation de hue au lieu de palette
    plt.figure(figsize=(10,6))
    sns.barplot(data=coef_df.head(10), x='Coefficient', y='Feature', hue='Coefficient', palette='coolwarm', legend=False)
    plt.title('Top 10 coefficients (importance) - Régression logistique')
    plt.tight_layout()
    plt.show()

elif hasattr(final_model, 'feature_importances_'):
    importances = final_model.feature_importances_
    feature_names = numeric_features + list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features))
    imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
    imp_df = imp_df.sort_values('Importance', ascending=False)
    print("\n=== Importance des features (XGBoost) ===")
    print("Top 10 features :")
    print(imp_df.head(10).to_string(index=False))
    
    plt.figure(figsize=(10,6))
    sns.barplot(data=imp_df.head(10), x='Importance', y='Feature', hue='Importance', palette='viridis', legend=False)
    plt.title('Top 10 importances - XGBoost')
    plt.tight_layout()
    plt.show()

print("\nÉvaluation terminée. Le modèle est prêt pour la prédiction sur de nouvelles données.")

In [ ]:
# -*- coding: utf-8 -*-
# Heart Disease Dataset - Étape 8 : Déploiement (version corrigée)
# Kaggle Notebook ready

import numpy as np
import pandas as pd
import joblib
import os

# ---------------------------
# 1. CHARGER LE MODÈLE FINAL ET RECONSTRUIRE LE PRÉPROCESSEUR
# ---------------------------
final_model = joblib.load('/kaggle/working/final_model_optimized.pkl')
print("Modèle chargé :", type(final_model).__name__)

# Recharger les données originales pour entraîner le préprocesseur
df_original = pd.read_csv('/kaggle/input/datasets/benafialoua/heartdisease/heart_disease.csv')

# Nettoyage (identique aux étapes précédentes)
critical_cols = ['age', 'trestbps', 'chol', 'thalach']
for col in critical_cols:
    df_original = df_original[df_original[col].notna()]
df_original = df_original[np.floor(df_original['age']) == df_original['age']]
df_original = df_original[np.floor(df_original['trestbps']) == df_original['trestbps']]
df_original = df_original[np.floor(df_original['chol']) == df_original['chol']]
df_original = df_original[np.floor(df_original['thalach']) == df_original['thalach']]
for col in critical_cols:
    df_original[col] = df_original[col].astype(int)
df_original = df_original[(df_original['chol'] >= 100) & (df_original['chol'] <= 600)]
df_original = df_original[(df_original['trestbps'] >= 80) & (df_original['trestbps'] <= 200)]
df_original = df_original[(df_original['oldpeak'] >= 0) & (df_original['oldpeak'] <= 6)]

X_clean = df_original.drop(columns=['num', 'target_binary'])

# Préprocesseur (standardisation + one-hot encoding)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
categorical_features = ['cp', 'restecg', 'slope', 'ca', 'thal', 'sex', 'fbs', 'exang']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ])
preprocessor.fit(X_clean)

# Sauvegarder modèle et préprocesseur
joblib.dump(final_model, '/kaggle/working/deployment_model.pkl')
joblib.dump(preprocessor, '/kaggle/working/deployment_preprocessor.pkl')
print("Modèle et préprocesseur sauvegardés dans /kaggle/working/")

# ---------------------------
# 2. FONCTION DE PRÉDICTION PAR LOT (CSV)
# ---------------------------
def predict_batch(input_csv_path, output_csv_path=None):
    """
    Prédit sur un fichier CSV contenant les mêmes colonnes (sans 'num' ni 'target_binary')
    et retourne un DataFrame avec les prédictions.
    """
    new_data = pd.read_csv(input_csv_path)
    X_new = new_data.drop(columns=['num', 'target_binary'], errors='ignore')
    X_processed = preprocessor.transform(X_new)
    predictions = final_model.predict(X_processed)
    probas = final_model.predict_proba(X_processed)[:, 1]
    new_data['prediction'] = predictions
    new_data['probabilite_maladie'] = probas
    if output_csv_path:
        new_data.to_csv(output_csv_path, index=False)
        print(f"Prédictions sauvegardées dans {output_csv_path}")
    return new_data

# ---------------------------
# 3. GÉNÉRATION DU CODE POUR L'API FASTAPI
# ---------------------------
api_code = '''# app.py - API FastAPI
import numpy as np
import pandas as pd
import joblib
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title="Heart Disease Predictor")

# Chargement du modèle et du préprocesseur
model = joblib.load("deployment_model.pkl")
preprocessor = joblib.load("deployment_preprocessor.pkl")

class PatientData(BaseModel):
    age: float
    sex: int
    cp: int
    trestbps: float
    chol: float
    fbs: int
    restecg: int
    thalach: float
    exang: int
    oldpeak: float
    slope: int
    ca: float
    thal: float

@app.post("/predict")
def predict(patient: PatientData):
    try:
        input_df = pd.DataFrame([patient.dict()])
        X = preprocessor.transform(input_df)
        pred = model.predict(X)[0]
        proba = model.predict_proba(X)[0, 1]
        return {"prediction": int(pred), "probabilite_maladie": float(proba)}
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

@app.get("/health")
def health():
    return {"status": "ok"}
'''

# ---------------------------
# 4. GÉNÉRATION DU CODE POUR L'APPLICATION STREAMLIT
# ---------------------------
streamlit_code = '''# streamlit_app.py
import streamlit as st
import numpy as np
import pandas as pd
import joblib

st.set_page_config(page_title="Prédiction Maladie Cardiaque", layout="centered")
st.title("🫀 Prédiction du risque de maladie cardiaque")
st.markdown("Renseignez les informations cliniques du patient ci-dessous.")

@st.cache_resource
def load_models():
    model = joblib.load("deployment_model.pkl")
    preprocessor = joblib.load("deployment_preprocessor.pkl")
    return model, preprocessor

model, preprocessor = load_models()

with st.form("patient_form"):
    col1, col2 = st.columns(2)
    with col1:
        age = st.number_input("Âge", min_value=20, max_value=100, value=55)
        sex = st.selectbox("Sexe", ["Femme", "Homme"])
        cp = st.selectbox("Douleur thoracique", 
                          [("Angine typique",1), ("Angine atypique",2), 
                           ("Douleur non angineuse",3), ("Asymptomatique",4)], format_func=lambda x: x[0])
        trestbps = st.number_input("Pression artérielle repos (mm Hg)", min_value=80, max_value=200, value=120)
        chol = st.number_input("Cholestérol (mg/dl)", min_value=100, max_value=600, value=200)
    with col2:
        fbs = st.selectbox("Glycémie à jeun > 120 mg/dl", ["Non", "Oui"])
        restecg = st.selectbox("ECG repos", [("Normal",0), ("Anomalie ST-T",1), ("Hypertrophie ventriculaire",2)], format_func=lambda x: x[0])
        thalach = st.number_input("Fréquence cardiaque max", min_value=70, max_value=220, value=150)
        exang = st.selectbox("Angine d'effort", ["Non", "Oui"])
        oldpeak = st.number_input("Dépression ST (oldpeak)", min_value=0.0, max_value=6.0, value=1.0, step=0.1)
        slope = st.selectbox("Pente ST", [("Montante",1), ("Plate",2), ("Descendante",3)], format_func=lambda x: x[0])
        ca = st.selectbox("Nb vaisseaux colorés (0-3)", [0,1,2,3])
        thal = st.selectbox("Thalassémie", [("Normal",3), ("Défaut réversible",7), ("Défaut fixé",6)], format_func=lambda x: x[0])
    
    submitted = st.form_submit_button("Prédire le risque")

if submitted:
    sex_val = 1 if sex == "Homme" else 0
    cp_val = cp[1]
    fbs_val = 1 if fbs == "Oui" else 0
    restecg_val = restecg[1]
    exang_val = 1 if exang == "Oui" else 0
    slope_val = slope[1]
    thal_val = thal[1]
    
    input_data = pd.DataFrame([{
        'age': age, 'sex': sex_val, 'cp': cp_val, 'trestbps': trestbps,
        'chol': chol, 'fbs': fbs_val, 'restecg': restecg_val, 'thalach': thalach,
        'exang': exang_val, 'oldpeak': oldpeak, 'slope': slope_val, 'ca': ca,
        'thal': thal_val
    }])
    
    X = preprocessor.transform(input_data)
    proba = model.predict_proba(X)[0, 1]
    pred = model.predict(X)[0]
    
    st.subheader("Résultat")
    if pred == 1:
        st.error(f"⚠️ Risque élevé de maladie cardiaque (probabilité : {proba:.2%})")
    else:
        st.success(f"✅ Risque faible de maladie cardiaque (probabilité : {proba:.2%})")
    
    st.info("Ceci est une aide à la décision clinique. Consultez toujours un médecin.")
'''

# ---------------------------
# 5. SAUVEGARDER LES FICHIERS DE DÉPLOIEMENT
# ---------------------------
deploy_dir = '/kaggle/working/deployment_package'
os.makedirs(deploy_dir, exist_ok=True)

# Copier les modèles
joblib.dump(final_model, f'{deploy_dir}/deployment_model.pkl')
joblib.dump(preprocessor, f'{deploy_dir}/deployment_preprocessor.pkl')

# Sauvegarder les scripts
with open(f'{deploy_dir}/api_app.py', 'w') as f:
    f.write(api_code)
with open(f'{deploy_dir}/streamlit_app.py', 'w') as f:
    f.write(streamlit_code)

# Créer un fichier requirements.txt
requirements = """numpy==1.24.3
pandas==2.0.3
scikit-learn==1.3.0
xgboost==1.7.6
joblib==1.3.2
fastapi==0.100.0
uvicorn==0.23.2
streamlit==1.25.0
pydantic==2.1.1
"""
with open(f'{deploy_dir}/requirements.txt', 'w') as f:
    f.write(requirements)

# Créer un fichier README (correctement formaté)
readme_content = (
    "# Heart Disease Prediction - Déploiement\n\n"
    "Ce dossier contient tout le nécessaire pour déployer le modèle de prédiction du risque de maladie cardiaque.\n\n"
    "## Fichiers\n"
    "- `deployment_model.pkl` : modèle entraîné\n"
    "- `deployment_preprocessor.pkl` : préprocesseur (standardisation + encodage)\n"
    "- `api_app.py` : API REST (FastAPI)\n"
    "- `streamlit_app.py` : interface utilisateur (Streamlit)\n"
    "- `requirements.txt` : dépendances Python\n\n"
    "## Utilisation\n\n"
    "### API\n"
    "```bash\n"
    "pip install -r requirements.txt\n"
    "uvicorn api_app:app --host 0.0.0.0 --port 8000\n"
    "```\n\n"
    "### Application web\n"
    "```bash\n"
    "streamlit run streamlit_app.py\n"
    "```\n\n"
    "### Prédiction par lot\n"
    "Exemple dans le notebook (fonction predict_batch).\n"
)
with open(f'{deploy_dir}/README.md', 'w') as f:
    f.write(readme_content)

print(f"\n✅ Tous les fichiers de déploiement sont dans : {deploy_dir}")
print("\nContenu du dossier :")
for file in os.listdir(deploy_dir):
    print(f"  - {file}")

# ---------------------------
# 6. TEST RAPIDE (optionnel)
# ---------------------------
sample = pd.DataFrame([{
    'age': 55, 'sex': 1, 'cp': 2, 'trestbps': 130, 'chol': 250,
    'fbs': 0, 'restecg': 0, 'thalach': 150, 'exang': 0, 'oldpeak': 1.2,
    'slope': 2, 'ca': 0, 'thal': 3
}])
sample.to_csv(f'{deploy_dir}/sample_input.csv', index=False)
result = predict_batch(f'{deploy_dir}/sample_input.csv', f'{deploy_dir}/sample_output.csv')
print("\nTest avec un patient fictif :")
print(result[['age', 'chol', 'prediction', 'probabilite_maladie']].head())

print("\nDéploiement prêt. Téléchargez le dossier 'deployment_package' depuis /kaggle/working/.")